# 5.06 — SRR3997504 + SRR4099955 salmon / L1EM 5'RACE experiment

Port of legacy `4.09-ach-albert_exp_SRR3997504_SRR4099955` (cells 16/20) into the R/ggplot
reproducibility scheme. **Stage A** (`scripts/aggregate_srr_l1em_exp.py`, run on Grace)
joins five per-L1-locus quantifications on the locus id (`region`) and writes the tidy
`results/srr_experiment/l1em_salmon_fpm.csv`
(`region, chrom, chromStart, chromEnd, target, category, strand, race5, method, value`).
**Stage B** (this notebook) renders `SRR3997504_SRR4099955_salmon_l1em_exp` to
`reports/figures/srr_experiment/`.

Each panel regresses one method's per-locus estimate (`value`) against the 5'RACE
ground-truth percentage (`race5`): **(A) AlbertSalmon, (B) Salmon** (salmon TPM,
classifier-filtered vs plain) and **(C) AlbertEM, (D) L1EM** (EM RPM). The two
quantifier families report different units, so each row carries its own y-axis title
(`TPM` for the Salmon row, `RPM` for the EM row) under a single shared `Estimated
abundance` label. Points are coloured by
target subfamily (L1HS / L1PA2); a red `lm` fit + R² label per panel; the largest-residual
locus is circled. Same data, panels, per-panel filter and statistics as the legacy PNG;
verify against `notebooks/images/SRR3997504_SRR4099955_salmon_l1em_exp.png`.


In [ ]:
## Reproducibility & environment capture -------------------------------
set.seed(3469)
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(jsonlite)
  library(gtable)   # compose the TPM and RPM rows with their own y-axis titles
  library(grid)
})
options(repr.plot.width = 14, repr.plot.height = 12)
theme_set(theme_bw(base_size = 28.125))   # legacy 4.09 fonts (plt.rc size 18; bold titles/labels)


In [ ]:
## Paths, provenance, palette ------------------------------------------
results_dir <- file.path("..", "results", "srr_experiment")
figures_dir <- file.path("..", "reports", "figures", "srr_experiment")
dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)
cat("results_dir:", results_dir, "\n"); cat("figures_dir:", figures_dir, "\n")

manifest_path <- file.path(results_dir, "manifest.json")
if (file.exists(manifest_path)) {
  manifest <- fromJSON(manifest_path)
  cat("git_commit:", manifest$git_commit, " timestamp:", manifest$timestamp_utc, "\n")
  if (!is.null(manifest$seed)) stopifnot(manifest$seed == 3469)
} else warning("manifest.json not found")

# Target subfamily palette (fixed; consistent with 5.03/5.04). Red #D62728 stays reserved for
# the lm fit line; green #2CA02C marks the largest-residual locus (legacy highlight).
target_palette <- c("L1HS" = "#1F77B4", "L1PA2" = "#FF7F0E")
target_shape   <- c("L1HS" = 16, "L1PA2" = 4)
panel_order    <- c("AlbertSalmon", "Salmon", "AlbertEM", "L1EM")
# Native output scale of each method; drives the per-row y-axis title.
unit_of        <- c(AlbertSalmon = "TPM", Salmon = "TPM", AlbertEM = "RPM", L1EM = "RPM")
source(file.path("..", "scripts/R/trap_plot_helpers.R"))
save_fig <- trap_save_fig(figures_dir, default_width = 13, default_height = 11)


In [ ]:
## Load the tidy table + validation checks ----------------------------
exp_path <- file.path(results_dir, "l1em_salmon_fpm.csv")
stopifnot("l1em_salmon_fpm.csv not found — run the Stage-A translator" = file.exists(exp_path))
E <- fread(exp_path)

req <- c("region", "target", "category", "race5", "method", "value")
stopifnot("missing required columns" = all(req %in% names(E)))
stopifnot("race5 must be finite and >= 0" = all(is.finite(E$race5) & E$race5 >= 0))
stopifnot("value (FPM/TPM) must be >= 0 where present" = all(E$value >= -1e-9 | is.na(E$value)))

# legacy cell-20 selection: expressed L1HS/L1PA2 loci (category > 0).
E <- E[grepl("^L1HS|^L1PA2", target) & category > 0]
E[, target := ifelse(grepl("^L1HS", target), "L1HS", "L1PA2")]
present <- intersect(panel_order, unique(E$method))
if (length(present) < length(panel_order))
  warning(sprintf("missing method(s): %s", paste(setdiff(panel_order, present), collapse = ", ")))
stopifnot("expected targets L1HS/L1PA2 present" = all(c("L1HS", "L1PA2") %in% E$target))
cat("Loaded", nrow(E), "method-locus rows;", uniqueN(E$region), "loci; methods:",
    paste(present, collapse = ", "), "\n")


In [ ]:
## Figure: SRR3997504_SRR4099955_salmon_l1em_exp ----------------------
# ---- Locus filter: single declared definition for all published panels -------
# Provenance: the estimate-side cut was developed and calibrated on the L1EM/EM *RPM* columns.
# The reference-side arm keeps loci carrying more than 1% of total 5'RACE signal (`race5` is
# already a percentage of total signal), i.e. a reference-side activity threshold.
# UNITS: `value` pools TPM (AlbertSalmon, Salmon) with RPM (AlbertEM, L1EM) in one column, so
# RACE5_MIN_PCT is unit-free while VALUE_MAX_RPM is an RPM-scale cut.
# separately in trap/analysis/robustness.py and do NOT alter these published panels.
RACE5_MIN_PCT <- 1    # reference-side activity threshold, % of total 5'RACE signal
VALUE_MAX_RPM <- 10   # estimate-side support cut, L1EM-calibrated, RPM scale
# -------------------------------------------------------------------------------------------
# Per method (legacy plot_regression_subplots): drop NA (race5,value); keep loci with
# race5 > RACE5_MIN_PCT OR value < VALUE_MAX_RPM (removes high-estimate / near-zero-RACE
# false positives); fit value ~ race5; R^2; flag the largest-residual locus.
# The green circle marks `maxdev` = the locus with the LARGEST RESIDUAL from that panel's own
# fit. This is not the most influential observation: removing it raises R^2, whereas removing
# the high-leverage locus at race5 = 12.21 (chr22:28663283-28669315) lowers every method's R^2. Fit/R^2 and the max-residual locus
# use the raw values; a small deterministic jitter is applied only for display (matches the
# legacy noise_scale = 0.01). The highlight circle is placed on the *jittered* coordinates of
# the max-residual locus so it is centred on the displayed point rather than its un-jittered
# position.
build_panel <- function(m) {
  d <- E[method == m & is.finite(race5) & is.finite(value)]
  d <- d[race5 > RACE5_MIN_PCT | value < VALUE_MAX_RPM]
  if (nrow(d) < 3) { warning(sprintf("%s: <3 loci after filter — skipped", m)); return(NULL) }
  fit <- lm(value ~ race5, data = d)
  d[, pred := predict(fit)]
  xr <- max(diff(range(d$race5)), 1e-9); yr <- max(diff(range(d$value)), 1e-9)
  d[, race5_j := race5 + rnorm(.N, 0, 0.01 * xr)]
  d[, value_j := value + rnorm(.N, 0, 0.01 * yr)]
  list(pts = d[, .(method = m, region, target, race5, value, race5_j, value_j)],
       fit = data.table(method = m, slope = coef(fit)[2], intercept = coef(fit)[1],
                        r2 = summary(fit)$r.squared),
       maxdev = d[which.max(abs(value - pred)), .(method = m, region, race5_j, value_j)])
}

panels <- lapply(present, build_panel); panels <- panels[!vapply(panels, is.null, logical(1))]
pts    <- rbindlist(lapply(panels, `[[`, "pts"))
fits   <- rbindlist(lapply(panels, `[[`, "fit"))
maxdev <- rbindlist(lapply(panels, `[[`, "maxdev"))
# Panel strip titles like the legacy: "(A) AlbertSalmon vs 5'RACE" (bold).
strip_lab <- setNames(sprintf("(%s) %s vs 5'RACE", LETTERS[seq_along(present)], present), present)
for (dt in list(pts, fits, maxdev)) dt[, method := factor(method, levels = present)]

# The four panels carry two different output scales, so each row of the figure gets its
# own y-axis title: Salmon-based methods report TPM, EM-based methods RPM. "Estimated
# abundance" is drawn once, outside both rows, as the quantity the two scales share.
# MDPI style: thousands separators only from five digits up ("10,000", but "8000").
build_row <- function(ms, ylab, xlab) {
  ggplot(pts[method %in% ms], aes(race5_j, value_j)) +
    geom_point(aes(colour = target, shape = target), size = 2.4, alpha = 0.85) +
    geom_abline(data = fits[method %in% ms], aes(slope = slope, intercept = intercept),
                colour = "#D62728", linewidth = 1) +
    geom_point(data = maxdev[method %in% ms], aes(race5_j, value_j), shape = 21, size = 6,
               colour = "#2CA02C", fill = NA, stroke = 1.4, inherit.aes = FALSE) +
    geom_text(data = fits[method %in% ms],
              aes(x = -Inf, y = Inf, label = sprintf("R^2 == %.2f", r2)),
              parse = TRUE, hjust = -0.12, vjust = 1.6, size = 9.375, inherit.aes = FALSE) +
    facet_wrap(~method, ncol = 2, scales = "free", labeller = as_labeller(strip_lab)) +
    scale_colour_manual(values = target_palette, name = "Target") +
    scale_shape_manual(values = target_shape, name = "Target") +
    scale_y_continuous(labels = comma_from_5_digits) +
    labs(x = xlab, y = ylab) +
    theme(legend.position = "none",
          strip.text = element_text(size = 18, face = "bold"),
          axis.title = element_text(size = 28.125, face = "bold"))
}
# One row per unit scale, in panel_order; the x-axis title is drawn once, under the last row.
rows <- split(present, unit_of[present])[unique(unit_of[present])]
subs <- lapply(seq_along(rows), function(i)
  build_row(rows[[i]], names(rows)[i], if (i == length(rows)) "5'RACE" else NULL))

# Stack the rows on a common width so the panels stay aligned, then draw the legend and the
# shared y-axis title once, spanning both rows.
leg    <- gtable_filter(ggplotGrob(subs[[1]] + theme(legend.position = "right")),
                        "guide-box-right")
grobs  <- lapply(subs, ggplotGrob)
w      <- Reduce(unit.pmax, lapply(grobs, `[[`, "widths"))
grobs  <- lapply(grobs, function(g) { g$widths <- w; g })
fig    <- Reduce(function(a, b) gtable:::rbind.gtable(a, b, size = "first"), grobs)
fig    <- gtable_add_cols(fig, sum(leg$widths))
fig    <- gtable_add_grob(fig, leg, t = 1, b = nrow(fig), l = ncol(fig), clip = "off",
                          name = "legend")
ylab_outer <- textGrob("Estimated abundance", rot = 90,
                       gp = gpar(fontsize = 28.125, fontface = "bold"))
fig <- gtable_add_cols(fig, grobWidth(ylab_outer) + unit(6, "pt"), pos = 0)
fig <- gtable_add_grob(fig, ylab_outer, t = 1, b = nrow(fig), l = 1, clip = "off",
                       name = "ylab-outer")

save_fig(fig, "SRR3997504_SRR4099955_salmon_l1em_exp", 13, 11)
print(fits[, .(method, r2 = round(r2, 3))])
grid.newpage(); grid.draw(fig)


In [ ]:
sessionInfo()